In [2]:
import xarray as xr
import a6
import matplotlib.pyplot as plt

import pathlib


plots = pathlib.Path("/p/project/hclimrep/emmerich1/plots/paper-1")

In [ ]:
!ls /p/scratch/hclimrep/emmerich1/data/ecmwf_era5

In [ ]:
paths = list(
    sorted(pathlib.Path("/p/scratch/hclimrep/emmerich1/data/ecmwf_era5").rglob(
        "**/*.nc"
    ))
)
paths

In [ ]:
ds = xr.open_mfdataset(
    paths, concat_dim="valid_time", combine="nested"
).rename(
    {"valid_time": "time", "pressure_level": "level"}
).drop_vars(["expver", "number"])
ds

In [ ]:
!mkdir -p /p/project/hclimrep/emmerich1/data/ecmwf_era5

In [ ]:
%%time
ds.astype("float32").to_netcdf("/p/project/hclimrep/emmerich1/data/ecmwf_era5/1964-2023-12UTC-300-500-700-850-950-hPa-z-r-t-u-v.nc")

In [3]:
!ls -al --block-size=G /p/project/hclimrep/emmerich1/data/ecmwf_era5

total 395G
drwxr-sr-x 2 emmerich1 17154   1G Mar 24 10:19 .
drwxr-sr-x 5 emmerich1 17154   1G Mar 21 22:46 ..
-rw-r--r-- 1 emmerich1 17154   2G Mar 24 10:12 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized-float32.nc
-rw-r--r-- 1 emmerich1 17154   2G Mar 24 10:19 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized-preprocessed-for-pca-float32.nc
-rw-r--r-- 1 emmerich1 17154   3G Mar 21 22:24 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized-preprocessed-for-pca.nc
-rw-r--r-- 1 emmerich1 17154   1G Mar 19 11:53 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized.nc
-rw-r--r-- 1 emmerich1 17154   2G Mar 24 10:03 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-float32.nc
-rw-r--r-- 1 emmerich1 17154   3G Mar 19 11:22 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-preprocessed-for-pca.nc
-rw-r--r-- 1 emmerich1 17154   1G Mar 19 05:37 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v.nc
-rw-r--r-- 1 emmerich1 17154  64G Mar 24 10:12 1964-2023-12UTC-300-500-700-850

In [ ]:
def deseasonalized_by_rolling_mean(data: xr.Dataset, days: int) -> xr.Dataset:
    return data.rolling(time=days, center=True, min_periods=1).mean()

def deseasonalize_by_trend(data: xr.Dataset) -> tuple[xr.Dataset, xr.DataArray]:
    time = data.time
    weighted = a6.features.methods.weighting.weight_by_latitudes(
        data,
        latitudes="latitude",
        use_sqrt=True,
        non_functional=True,
    )
    mean = weighted.mean(("latitude", "longitude"))
    fit = mean.polyfit(dim=time.name, deg=1)
    trend = xr.polyval(coord=time, coeffs=fit.polyfit_coefficients)
    detrended = data - (trend - trend[0])
    return detrended, trend, fit

def deseasonalize(data: xr.Dataset) -> xr.Dataset:
    for i, level in enumerate(data.level):
        data_level = data.sel(level=level)
        for name, field in data_level.data_vars.items():
            smoothed = deseasonalized_by_rolling_mean(field, days=60)
            detrended, _, _ = deseasonalize_by_trend(smoothed)
            data_level.update({name: detrended})
        data[{"level": i}] = data_level
    return data

In [ ]:
%%time

ds_deseasonalized = deseasonalize(ds)
ds_deseasonalized

In [ ]:
%%time

ds_deseasonalized.astype("float32").to_netcdf("/p/project/hclimrep/emmerich1/data/ecmwf_era5/1964-2023-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized.nc")

In [8]:
!ls -al --block-size=G /p/project/hclimrep/emmerich1/data/ecmwf_era5/

total 331G
drwxr-sr-x 2 emmerich1 17154   1G Mar 24 10:12 .
drwxr-sr-x 5 emmerich1 17154   1G Mar 21 22:46 ..
-rw-r--r-- 1 emmerich1 17154   2G Mar 24 10:12 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized-float32.nc
-rw-r--r-- 1 emmerich1 17154   3G Mar 21 22:24 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized-preprocessed-for-pca.nc
-rw-r--r-- 1 emmerich1 17154   1G Mar 19 11:53 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized.nc
-rw-r--r-- 1 emmerich1 17154   2G Mar 24 10:03 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-float32.nc
-rw-r--r-- 1 emmerich1 17154   3G Mar 19 11:22 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-preprocessed-for-pca.nc
-rw-r--r-- 1 emmerich1 17154   1G Mar 19 05:37 1964-12UTC-300-500-700-850-950-hPa-z-r-t-u-v.nc
-rw-r--r-- 1 emmerich1 17154  64G Mar 24 10:12 1964-2023-12UTC-300-500-700-850-950-hPa-z-r-t-u-v-deseasonalized-float32.nc
-rw-r--r-- 1 emmerich1 17154 128G Mar 19 12:26 1964-2023-12UTC-300-500-700-850-950-hPa-z-r-t-u

## Create deseasonalization plot for temperature at 500 hPa level

In [ ]:
%%time

def weighted_spatial_mean(data: xr.Dataset) -> xr.Dataset:
    weighted = a6.features.methods.weighting.weight_by_latitudes(
        data,
        latitudes="latitude",
        use_sqrt=True,
        non_functional=True,
    )
    return weighted.mean(("latitude", "longitude"))

time = ds.time

ds_level = ds.sel(level=500)
field = ds_level["t"]
smoothed = deseasonalized_by_rolling_mean(field, days=60)
detrended, trend, coeffs = deseasonalize_by_trend(smoothed)

fig, ax = plt.subplots(figsize=(12,4))

weighted_spatial_mean(field).plot(label="field", color="black", ax=ax)
weighted_spatial_mean(smoothed).plot(label="smoothed", color="tab:blue", ax=ax)
trend.plot(label="trend", linestyle="--", color="tab:red", ax=ax)
weighted_spatial_mean(detrended).plot(label="smoothed+detrended", color="tab:orange", ax=ax)
ax.set_ylabel(r"$t_{500}\,[K]$")
ax.set_xlabel("Time")
ax.set_title("")
ax.set_xlim(time[5 * 365], time[10 * 365])
plt.legend()
plt.savefig(plots / "figXX.png", dpi=300, bbox_inches="tight")

In [ ]:
fit.compute().values

In [ ]:
3.418e-18 * 365 * 200